# Notebook 03: Feature Engineering & Gold Layer Verification
This notebook verifies the engineered features, StringIndexer encodings, VectorAssembler representation, and data integrity in the Gold Parquet layer.

In [ ]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

sys.path.append('../src')
from utils import create_spark_session, get_project_root, load_json

# Cell 1: Load Gold Parquet
spark = create_spark_session(app_name="03-feature-verification")
root_dir = get_project_root()
gold_path = root_dir / "data" / "gold" / "gold_features.parquet"
df_gold = spark.read.parquet(str(gold_path))
print(f"Loaded Gold Feature dataset from: {gold_path}")

In [ ]:
# Cell 2: Display Feature Columns
feature_meta_path = root_dir / "results" / "feature_names.json"
if feature_meta_path.exists():
    meta = load_json(feature_meta_path)
    print("Feature Metadata:")
    for k, v in meta.items():
        print(f"  {k}: {v}")
else:
    print("Columns:", df_gold.columns)

In [ ]:
# Cell 3: Display Row Count
gold_count = df_gold.count()
print(f"Total Gold Feature Rows: {gold_count:,}")

In [ ]:
# Cell 4: Sample Feature Vectors (First 10 Rows)
df_gold.select("features", "price").show(10, truncate=False)

In [ ]:
# Cell 5: Check for Nulls in Final Feature Set
feature_cols = [c for c in df_gold.columns if c != "features"]
null_exprs = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in feature_cols]
df_gold.select(null_exprs).show(truncate=False)

In [ ]:
# Cell 6: Summary Statistics for Numerical Features (year, month, quarter)
df_gold.select("year", "month", "quarter", "price").describe().show(truncate=False)

In [ ]:
# Cell 7: Verify StringIndexer Outputs are Numeric
indexed_cols = ["property_type_idx", "new_build_idx", "duration_idx", "county_idx", "district_idx", "town_idx"]
for col_name in indexed_cols:
    dtype = dict(df_gold.dtypes).get(col_name)
    print(f"Column '{col_name}' DataType: {dtype}")
    assert "double" in dtype.lower() or "int" in dtype.lower() or "float" in dtype.lower(), f"{col_name} is not numeric!"
print("All StringIndexer columns verified as numeric!")